# LatentController — Training Pipeline

Looping transformer with persistent trie-indexed memory, adaptive computation time (ACT), and streaming token processing.

**Phases:**
1. Baseline language model (TinyStories)
2. Address head contrastive pretraining (frozen backbone)
3. Memory integration training (frozen address heads)
4. ACT training with ponder curriculum (frozen address heads)
5. Unified streaming — all systems active (address heads unfrozen at 0.3× LR)

**Requirements:** A100 40GB GPU (Colab Pro)

In [ ]:
!git clone https://github.com/kaaninel/LatentController.git
%cd LatentController
!pip install -q torch datasets tokenizers numpy psutil

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login, upload_folder, snapshot_download
login()  # paste your HF token (free account works)

In [ ]:
!git pull

## Download / Upload Checkpoints
Run the download cell to resume from a prior run. Run the upload cell after training to save progress.

In [ ]:
snapshot_download(
    repo_id="kaaninel/latentcontroller",
    local_dir="./",
    repo_type="model"
)

In [ ]:
upload_folder(
    folder_path="./checkpoints",
    repo_id="kaaninel/latentcontroller",
    path_in_repo="checkpoints",
    repo_type="model"
)
upload_folder(
    folder_path="./data_cache",
    repo_id="kaaninel/latentcontroller",
    path_in_repo="data_cache",
    repo_type="model"
)

## Training Phases
Run phases in order (2→3→4→5). Phase 1 is already complete on HuggingFace.

Each phase auto-detects hardware, loads the best prior checkpoint, and resumes if interrupted.

In [ ]:
# Phase 1: Baseline LM (already complete — run only if retraining from scratch)
!python run_colab.py --phase 1 \
    --checkpoint_dir ./checkpoints \
    --data_dir ./data_cache

In [ ]:
# Phase 2: Address head contrastive pretraining (~10K steps, ~1 hour)
!python run_colab.py --phase 2 \
    --checkpoint_dir ./checkpoints \
    --data_dir ./data_cache

In [ ]:
# Phase 3: Memory integration (2B tokens, longest phase)
!python run_colab.py --phase 3 \
    --checkpoint_dir ./checkpoints \
    --data_dir ./data_cache

In [ ]:
# Phase 4: ACT training with ponder curriculum (1B tokens)
!python run_colab.py --phase 4 \
    --checkpoint_dir ./checkpoints \
    --data_dir ./data_cache

In [ ]:
# Phase 5: Unified streaming — all systems active (2B tokens)
# Default: TinyStories. For NOOP training, add --context_column and --dataset_name
!python run_colab.py --phase 5 \
    --checkpoint_dir ./checkpoints \
    --data_dir ./data_cache

In [ ]:
# Phase 5 with custom QA dataset (NOOP targets from context/response split)
# Uncomment and modify for your dataset:
#!python run_colab.py --phase 5 \
#    --checkpoint_dir ./checkpoints \
#    --data_dir ./data_cache \
#    --dataset_name "squad" \
#    --text_column "answers" \
#    --context_column "context"

## Save Progress
Upload checkpoints to HuggingFace after each phase completes.

In [ ]:
upload_folder(
    folder_path="./checkpoints",
    repo_id="kaaninel/latentcontroller",
    path_in_repo="checkpoints",
    repo_type="model"
)
upload_folder(
    folder_path="./data_cache",
    repo_id="kaaninel/latentcontroller",
    path_in_repo="data_cache",
    repo_type="model"
)